# Problemas inversos: evidencia y falta de unicidad

**Nivel 1 · Prerrequisitos: ninguno; lectura: 30 minutes.**

## Objetivos

Para el modelo m, las observaciones d, el operador directo F y el error ε, escribimos d=F(m)+ε. Un buen ajuste restringe predicciones en la geometría de adquisición; no demuestra una estructura única ni geológicamente correcta. Esta lección distingue identidad del artefacto, validación numérica e interpretación de campo.

## Preparación

Use el entorno de notebooks Python 3.12 revisado del repositorio. Este notebook no instala dependencias. El artefacto del producto es sintético, no es dato de campo.

La celda siguiente verifica los bytes del resultado versionado y lee su recibo; no vuelve a ejecutar un solver.

In [ ]:
import hashlib, json
from pathlib import Path

# Resolver los datos desde la raíz del repositorio; el runner inicia en la carpeta de esta lección.
root = Path.cwd().parents[2]
caso = root / 'data/derived/m01-scientific-course/prism-case-0'
recibo = json.loads((caso / 'receipt.json').read_text(encoding='utf-8'))
hash_resultado = hashlib.sha256((caso / 'result.json').read_bytes()).hexdigest()
print({'caso': 'control sintético de prisma', 'sha256_resultado': hash_resultado, 'campos_recibo': sorted(recibo)})

## Datos

La solicitud, el resultado y el recibo son artefactos de control sintético creados para el producto. Un hash solo prueba identidad de bytes: no demuestra corrección física, convergencia, unicidad ni elegibilidad de campo. No se representa un levantamiento medido.

**Archivos:** `data/derived/m01-scientific-course/prism-case-0/request.json`, `result.json`, `receipt.json`.

## Método

Un objetivo habitual es φ(m)=½‖W_d(F(m)−d)‖²+βφ_m(m−m_ref). W_d representa la incertidumbre declarada de los datos; β y φ_m codifican supuestos previos. Mallas, cotas, referencias y geometrías de adquisición diferentes pueden ajustar el mismo d con modelos distintos. SimPEG es un marco de simulación e inversión ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); un artefacto reproducible aún requiere controles numéricos independientes antes de sostener una afirmación física.

El Jacobiano J=∂F/∂m relaciona un cambio pequeño de modelo con un cambio en datos, δd≈Jδm. Direcciones del espacio nulo son invisibles para esta adquisición; valores singulares pequeños indican combinaciones débilmente restringidas. Una predicción coincidente no demuestra recuperación geológica. Una estimación regularizada puede minimizar φ=½‖W_d(F(m)−d)‖²+β‖L(m−m_ref)‖². W_d escala residuales por errores declarados; L y β codifican preferencias. Si d,σ están en mGal, W_dr es adimensional. Cambiar malla o regularizador puede elegir otro modelo con ajuste similar. SimPEG presenta este marco de simulación y estimación de parámetros [Cockett et al. (2015)](https://doi.org/10.1016/j.cageo.2015.09.015).

El artefacto es un caso gravimétrico sintético creado, no un levantamiento. Su mapa residual muestra soporte espacial, pero no es validación independiente ni imagen del subsuelo; hereda geometría y ruido fijados. Un checksum verifica identidad de bytes; la validez numérica requiere recálculo y controles independientes. La figura separa entrenamiento de holdout externo y enmascara predicciones sin soporte.

## Resultados

El código solo verifica la identidad del artefacto. No recalcula un modelo directo ni resuelve una inversión. El control almacenado es sintético y no informa sobre geología de campo.

Use residuales guardados en coordenadas locales reales. El color es predicción menos observación en mGal; cuadrados huecos marcan holdout externo. Diagnostica el caso sintético, no un campo observado.


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Graficar residuales sintéticos persistidos y calcular RMSE por partición con soporte.
caso = Path.cwd().parents[2] / 'data/derived/m01-scientific-course/prism-case-0'
a = json.loads((caso / 'result.json').read_text(encoding='utf-8'))
xy = np.column_stack([a['geometry']['easting_m'], a['geometry']['northing_m']])
r = np.asarray(a['stations']['signed_predicted_minus_observed_mgal'], dtype=float)
covered = np.asarray(a['stations']['prediction_covered'], dtype=bool)
part = np.asarray(a['split']['partition'])
fig, ax = plt.subplots(figsize=(8.2,5.6), constrained_layout=True)
sc = ax.scatter(xy[covered,0],xy[covered,1],c=r[covered],cmap='coolwarm',s=34,edgecolors='0.25',linewidths=.3)
hold = covered & (part=='holdout')
ax.scatter(xy[hold,0],xy[hold,1],facecolors='none',edgecolors='black',marker='s',s=70,label='Holdout externo')
ax.set(xlabel='Este [m]',ylabel='Norte [m]',title='Caso sintético: residuales con soporte'); ax.set_aspect('equal',adjustable='datalim'); ax.legend()
fig.colorbar(sc,ax=ax,label='Predicción − observación $g_z$ [mGal]')
for split in ('train','holdout'):
 use=covered & (part==split); print(f'{split}: n={use.sum()}, RMSE={np.sqrt(np.mean(r[use]**2)):.5f} mGal')
plt.show()


## Resumen

El ajuste de datos, la recuperación del modelo y la interpretación geológica son afirmaciones distintas. Declare la física, geometría, errores, información previa y discretización que condicionan cada resultado inverso.


## Referencias

- https://doi.org/10.1016/j.cageo.2015.09.015